# EnergeticPricing — prévision du prix day-ahead français

Pipeline complet : chargement des données (météo par département, prévisions ENTSO-E, gaz TTF, capacités installées), entraînement avec validation temporelle, puis prévision d'une journée.

Prérequis : une clé ENTSO-E dans `.env` (cf. `.env.example`).

In [ ]:
from src.market.market import EnergyMarket

## Paramètres

- **Cache** : les téléchargements sont mis en cache dans `data/cache/`. Si tu repousses `END`, seule la période manquante est téléchargée (quelques minutes, pas tout l'historique).
- **`NUCLEAR`** : features nucléaires (production J-2, arrêts planifiés REMIT). Premier téléchargement ~20 min, sans gain mesuré sur 2024-2026 → désactivé par défaut.
- **`NEIGHBOURS`** : prix day-ahead des pays voisins. Inconnus avant l'enchère (fuite) → uniquement pour de l'analyse a posteriori.

In [ ]:
START = "2023-01-01"
END = "2026-05-01"
FREQ = "1h"                # "1h" ou "15min"
METEO_MODE = "department"  # "department" (météo par dép.) ou "national" (moyenne pondérée pop.)
NUCLEAR = False            # features nucléaires (lent au premier chargement)
NEIGHBOURS = False         # prix voisins (fuite : analyse a posteriori seulement)

# "LightGBM" (modèle principal), "LightGBM_hurdle" (deux régimes : prix <= 0 / > 0),
# "Simple" (Ridge), "RandomForest", "GradientBoosting", "TFT"
MODEL = "LightGBM"

## Chargement des données

In [ ]:
market = EnergyMarket(
    start=START, end=END, freq=FREQ, meteo_mode=METEO_MODE,
    nuclear=NUCLEAR, neighbours=NEIGHBOURS,
)
market.initialize()

## Entraînement
Validation rolling-origin (TimeSeriesSplit, 5 plis). Les modèles entraînés sont mis en cache dans `results/models/`.

In [ ]:
market.learn(MODEL, use_neighbour_prices=NEIGHBOURS)

## Prévision d'une journée
Jusqu'à J+3 ; le prix réel est affiché s'il est publié.

In [ ]:
market.predict_from_meteo("2026-04-28")